In [0]:
from pyspark.sql import functions as F

caminho_volume = "/Volumes/prf_acidentes1/bronze/raw_files/"

arquivos = {
    2023: "datatran2023.csv",
    2024: "datatran2024.csv",
    2025: "datatran2025.csv"
}

dataframes = []

for ano, nome_arquivo in arquivos.items():
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")  # <-- mudou: tudo como string
        .csv(caminho_volume + nome_arquivo)
        .withColumn("_ano_referencia", F.lit(ano))
        .withColumn("_arquivo_origem", F.lit(nome_arquivo))
        .withColumn("_data_ingestao", F.current_timestamp())
    )
    dataframes.append(df)

df_bronze = dataframes[0]
for df in dataframes[1:]:
    df_bronze = df_bronze.unionByName(df, allowMissingColumns=True)

print("Total de linhas:", df_bronze.count())
df_bronze.printSchema()

Total de linhas: 213451
root
 |-- id: string (nullable = true)
 |-- data_inversa: string (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: string (nullable = true)
 |-- km: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: string (nullable = true)
 |-- mortos: string (nullable = true)
 |-- feridos_leves: string (nullable = true)
 |-- feridos_graves: string (nullable = true)
 |-- ilesos: string (nullable = true)
 |-- ignorados: string (nullable = true)
 |-- feridos: s

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes1.bronze.acidentes_raw")

print("Tabela Bronze criada com sucesso!")

Tabela Bronze criada com sucesso!


In [0]:
display(spark.sql("SELECT * FROM prf_acidentes1.bronze.acidentes_raw LIMIT 10"))

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,_ano_referencia,_arquivo_origem,_data_ingestao
496519,2023-01-01,domingo,02:00:00,ES,101,114,SOORETAMA,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,1,0,1,0,0,0,1,1,"-19,09484877","-40,05095848",SPRF-ES,DEL04-ES,UOP01-DEL04-ES,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496543,2023-01-01,domingo,03:40:00,SP,116,"113,1",TAUBATE,Entrada inopinada do pedestre,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,5,1,0,0,0,4,0,2,"-23,0445658","-45,58259814",SPRF-SP,DEL02-SP,UOP02-DEL02-SP,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496590,2023-01-01,domingo,01:40:00,MT,163,1112,GUARANTA DO NORTE,Reação tardia ou ineficiente do condutor,Tombamento,NA,Plena Noite,Crescente,Ignorado,Simples,Curva;Declive,Não,2,0,0,1,0,2,1,3,"-9,70020602","-54,87588757",SPRF-MT,DEL06-MT,UOP03-DEL06-MT,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496610,2023-01-01,domingo,10:40:00,PR,376,"314,8",ORTIGUEIRA,Velocidade Incompatível,Tombamento,Sem Vítimas,Pleno dia,Crescente,Sol,Dupla,Curva,Não,2,0,0,0,1,2,0,3,"-23,985512","-51,083555",SPRF-PR,DEL07-PR,UOP02-DEL07-PR,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496659,2023-01-01,domingo,14:55:00,MG,116,"569,4",MANHUACU,Acumulo de água sobre o pavimento,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Declive;Curva,Não,4,0,0,2,1,1,2,3,"-20,10007457","-42,17884091",SPRF-MG,DEL06-MG,UOP03-DEL06-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496671,2023-01-01,domingo,15:45:00,MG,262,"569,8",CORREGO DANTA,Condutor Dormindo,Saída de leito carroçável,Sem Vítimas,Pleno dia,Decrescente,Nublado,Simples,Reta;Aclive,Sim,2,0,0,0,1,1,0,2,"-19,716043","-46,021922",SPRF-MG,DEL08-MG,UOP02-DEL08-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496673,2023-01-01,domingo,18:10:00,PR,116,152,MANDIRITUBA,Desrespeitar a preferência no cruzamento,Colisão transversal,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Simples,Interseção de Vias,Sim,5,0,1,0,3,1,1,4,"-25,862553","-49,362008",SPRF-PR,DEL01-PR,UOP03-DEL01-PR,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496686,2023-01-01,domingo,20:00:00,MG,381,"897,3",CAMBUI,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Plena Noite,Crescente,Ignorado,Dupla,Reta,Não,1,0,0,0,1,0,0,1,"-22,63961103","-46,08077997",SPRF-MG,DEL16-MG,UOP03-DEL16-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496709,2023-01-01,domingo,21:54:00,SP,116,"105,5",TAUBATE,Transitar na contramão,Colisão lateral mesmo sentido,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Dupla,Reta,Não,5,0,2,0,1,2,2,4,"-23,00757973","-45,51466613",SPRF-SP,DEL02-SP,UOP02-DEL02-SP,2023,datatran2023.csv,2026-09-21T23:15:21.200Z
496711,2023-01-01,domingo,21:50:00,BA,116,366,SERRINHA,Velocidade Incompatível,Colisão traseira,Sem Vítimas,Plena Noite,Decrescente,Garoa/Chuvisco,Simples,Reta,Não,2,0,0,0,2,0,0,2,"-11,72514","-38,98642914",SPRF-BA,DEL02-BA,UOP01-DEL02-BA,2023,datatran2023.csv,2026-09-21T23:15:21.200Z


In [0]:
display(spark.sql("SHOW TABLES IN prf_acidentes1.bronze"))

database,tableName,isTemporary
bronze,acidentes_raw,false


In [0]:
display(spark.sql("DESCRIBE TABLE prf_acidentes1.bronze.acidentes_raw"))

col_name,data_type,comment
id,string,null
data_inversa,string,null
dia_semana,string,null
horario,string,null
uf,string,null
br,string,null
km,string,null
municipio,string,null
causa_acidente,string,null
tipo_acidente,string,null
